# 08 - Hypothesis tests and feature selection
**Goal:** formal tests in the tutorial style (hypotheses, formula, statistic, p-value, decision, interpretation):
one-way ANOVA + Tukey HSD, Kruskal-Wallis, Chi-square independence, effect sizes; then filter vs wrapper
feature selection for separating genres.

**Inputs:** `data/processed/genre_tracks.parquet`, `data/processed/tracks.parquet`.

**Outputs:** `reports/tables/tab_08_*.csv`, figure `fig_08_*`, `results.json -> tests`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as ss
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import VarianceThreshold, f_classif, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sonicatlas import DATA, TAB, SEED, results, theme
from sonicatlas.stats import eta_squared, cramers_v

theme.apply()
gt = pd.read_parquet(DATA / "processed" / "genre_tracks.parquet")
a = pd.read_parquet(DATA / "processed" / "tracks.parquet", columns=["mode", "decade", "is_canonical"])
FOCUS = ["classical", "jazz", "country", "rock", "metal", "hip-hop", "edm", "pop"]
FEATS = ["danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "duration_min"]
f8 = gt[gt.track_genre.isin(FOCUS)].copy()

## Test 1 - One-way ANOVA: does mean danceability differ across the eight genres?
* H0: mu_1 = mu_2 = ... = mu_8;  H1: at least one genre mean differs.
* F = MS_between / MS_within = [SS_B / (k - 1)] / [SS_W / (N - k)];  alpha = 0.05.

In [2]:
groups = [f8.loc[f8.track_genre == g, "danceability"].to_numpy() for g in FOCUS]
k, N = len(groups), sum(len(g) for g in groups)
gm = np.concatenate(groups).mean()
ssb = sum(len(g) * (g.mean() - gm) ** 2 for g in groups)
ssw = sum(((g - g.mean()) ** 2).sum() for g in groups)
F_hand = (ssb / (k - 1)) / (ssw / (N - k))
F, p = ss.f_oneway(*groups)
eta2 = eta_squared(groups)
anova = {"k": k, "N": N, "SSB": ssb, "SSW": ssw, "df_between": k - 1, "df_within": N - k, "F": float(F), "F_hand": float(F_hand),
         "p": float(p), "eta_squared": eta2, "group_means": {g: float(x.mean()) for g, x in zip(FOCUS, groups)},
         "group_n": {g: len(x) for g, x in zip(FOCUS, groups)}}
print(anova)

{'k': 8, 'N': 4863, 'SSB': np.float64(61.270744396606624), 'SSW': np.float64(82.9875997060091), 'df_between': 7, 'df_within': 4855, 'F': 512.0721393477321, 'F_hand': 512.0721393477321, 'p': 0.0, 'eta_squared': 0.4247292922828973, 'group_means': {'classical': 0.3859876585928489, 'jazz': 0.4890572519083969, 'country': 0.5548181818181818, 'rock': 0.5487573099415205, 'metal': 0.48117672413793106, 'hip-hop': 0.7300522565320666, 'edm': 0.6556340057636888, 'pop': 0.5805889423076923}, 'group_n': {'classical': 867, 'jazz': 524, 'country': 946, 'rock': 342, 'metal': 232, 'hip-hop': 842, 'edm': 694, 'pop': 416}}


In [3]:
tk = pairwise_tukeyhsd(f8["danceability"], f8["track_genre"], alpha=0.05)
tk_df = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
tk_df.to_csv(TAB / "tab_08_01_tukey_hsd_danceability.csv", index=False)
n_sig = int(tk_df["reject"].sum())
print(f"{n_sig} of {len(tk_df)} genre pairs differ significantly")
tk_df.sort_values("meandiff", key=abs, ascending=False).head(8)

26 of 28 genre pairs differ significantly


,group1,group2,meandiff,p-adj,lower,upper,reject
2,classical,hip-hop,0.3441,0.0,0.3249,0.3632,True
1,classical,edm,0.2696,0.0,0.2495,0.2898,True
19,hip-hop,metal,-0.2489,0.0,-0.2783,-0.2195,True
18,hip-hop,jazz,-0.2410,0.0,-0.2631,-0.2189,True
5,classical,pop,0.1946,0.0,0.1710,0.2182,True
21,hip-hop,rock,-0.1813,0.0,-0.2067,-0.1559,True
8,country,hip-hop,0.1752,0.0,0.1565,0.1940,True
15,edm,metal,-0.1745,0.0,-0.2045,-0.1444,True


## Test 2 - Kruskal-Wallis (non-parametric check; danceability is bounded and not normal)

In [4]:
H, pk = ss.kruskal(*groups)
eps2 = (H - k + 1) / (N - k)
lev_W, lev_p = ss.levene(*groups)
print(H, pk, eps2, lev_W, lev_p)

2060.5260060477012 0.0 0.4229713709676007 21.325557177658474 1.6451050176912195e-28


## Test 3 - Chi-square test of independence: mode (major/minor) vs genre, and mode vs decade
* H0: mode and genre are independent;  chi2 = sum (O - E)^2 / E,  E = row total x column total / N.

In [5]:
ct_g = pd.crosstab(f8.track_genre, f8["mode"].map({1: "major", 0: "minor"}))
chi2_g, p_g, dof_g, exp_g = ss.chi2_contingency(ct_g)
v_g = cramers_v(ct_g, chi2_g)
ct_g.assign(major_pct=(ct_g["major"] / ct_g.sum(1) * 100).round(1)).to_csv(TAB / "tab_08_02_mode_by_genre.csv")
ac = a[a.is_canonical & a.decade.between(1950, 2010)]
ct_d = pd.crosstab(ac.decade, ac["mode"].map({1: "major", 0: "minor"}))
chi2_d, p_d, dof_d, _ = ss.chi2_contingency(ct_d)
v_d = cramers_v(ct_d, chi2_d)
ct_d.assign(major_pct=(ct_d["major"] / ct_d.sum(1) * 100).round(1)).to_csv(TAB / "tab_08_03_mode_by_decade.csv")
print(chi2_g, p_g, dof_g, v_g); print(chi2_d, p_d, dof_d, v_d)
ct_g.assign(major_pct=(ct_g["major"] / ct_g.sum(1) * 100).round(1))

539.3705158591665 2.733864961483976e-112 7 0.3330362183528481
3089.0249994003952 0.0 6 0.05430042449052129


C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\2315895306.py:4: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  ct_g.assign(major_pct=(ct_g["major"] / ct_g.sum(1) * 100).round(1)).to_csv(TAB / "tab_08_02_mode_by_genre.csv")
C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\2315895306.py:9: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  ct_d.assign(major_pct=(ct_d["major"] / ct_d.sum(1) * 100).round(1)).to_csv(TAB / "tab_08_03_mode_by_decade.csv")
C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\2315895306.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  ct_g.assign(major_pct=(ct_g["major"] / ct_g.sum(1) * 100).round(1))


mode,major,minor,major_pct
track_genre,,,
classical,661,206,76.2
country,845,101,89.3
edm,366,328,52.7
hip-hop,407,435,48.3
jazz,412,112,78.6
metal,135,97,58.2
pop,208,208,50.0
rock,233,109,68.1


In [6]:
ct_d.assign(major_pct=(ct_d["major"] / ct_d.sum(1) * 100).round(1))

C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\764727122.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  ct_d.assign(major_pct=(ct_d["major"] / ct_d.sum(1) * 100).round(1))


mode,major,minor,major_pct
decade,,,
1950,2175,749,74.4
1960,6029,1951,75.6
1970,11619,3828,75.2
1980,18352,7735,70.3
1990,100190,41465,70.7
2000,272074,124710,68.6
2010,296495,160275,64.9


## Feature selection: filter methods vs wrapper (RFE) for separating the eight genres

In [7]:
X = f8[FEATS].to_numpy(); y = f8.track_genre.to_numpy()
Xmm = MinMaxScaler().fit_transform(X)
Xz = StandardScaler().fit_transform(X)
var = pd.Series(VarianceThreshold().fit(Xmm).variances_, index=FEATS)
Fv, _ = f_classif(X, y)
mi = mutual_info_classif(Xz, y, random_state=SEED)
corr_max = pd.DataFrame(Xz, columns=FEATS).corr().abs().where(lambda m: m < 1).max()
rfe = RFE(LogisticRegression(max_iter=2000), n_features_to_select=1).fit(Xz, y)
fs = pd.DataFrame({"variance_minmax": var, "anova_F": Fv, "mutual_info": mi, "max_abs_corr_other": corr_max, "rfe_rank": rfe.ranking_}, index=FEATS)
ranks = pd.DataFrame({"variance": fs.variance_minmax.rank(ascending=False), "anova_F": fs.anova_F.rank(ascending=False),
                      "mutual_info": fs.mutual_info.rank(ascending=False), "rfe": fs.rfe_rank}).astype(int)
fs = fs.join(ranks.add_suffix("_rank").drop(columns="rfe_rank"))
fs.to_csv(TAB / "tab_08_04_feature_selection.csv")
rank_corr = ranks.corr("spearman")
print(rank_corr.round(2))
fs.sort_values("anova_F", ascending=False).round(3)

             variance  anova_F  mutual_info   rfe
variance         1.00     0.66         0.13  0.65
anova_F          0.66     1.00         0.65  0.90
mutual_info      0.13     0.65         1.00  0.58
rfe              0.65     0.90         0.58  1.00


,variance_minmax,anova_F,mutual_info,max_abs_corr_other,rfe_rank,variance_rank,anova_F_rank,mutual_info_rank
acousticness,0.138,1328.332,0.819,0.851,1,1,1,2
loudness,0.026,1198.037,0.821,0.846,2,7,2,1
energy,0.074,981.216,0.725,0.851,5,3,3,3
instrumentalness,0.093,792.462,0.451,0.712,3,2,4,8
danceability,0.032,512.072,0.588,0.518,4,5,5,6
speechiness,0.007,140.942,0.495,0.281,8,9,6,7
valence,0.059,34.739,0.409,0.463,6,4,7,9
tempo,0.029,27.364,0.637,0.236,9,6,8,5
duration_min,0.002,21.394,0.680,0.164,7,10,9,4
liveness,0.022,7.273,0.329,0.163,10,8,10,10


In [8]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(FEATS)); w = 0.2
for i, col in enumerate(["variance", "anova_F", "mutual_info", "rfe"]):
    ax.bar(x + (i - 1.5) * w, len(FEATS) + 1 - ranks[col], w, label=col, color=theme.OKABE_ITO[i], edgecolor="k", lw=0.4, hatch=["", "//", "..", "xx"][i])
ax.set_xticks(x, FEATS, rotation=35, ha="right")
top = ranks.sum(1).sort_values().index[:2]
ax.set(title=f"{top[0].capitalize()} and {top[1]} rank highest for separating genres in filter and wrapper methods",
       ylabel="Importance (11 - rank)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.32))
theme.save(fig, "fig_08_01_feature_selection_ranks")

C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\2019667221.py:6: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  top = ranks.sum(1).sort_values().index[:2]


WindowsPath('D:/VIT/SEM 3/EDA/project/Team_05_25BDS0318_25BDS0308_SonicAtlas/reports/figures/fig_08_01_feature_selection_ranks.png')

In [9]:
results.save("tests", {
    "anova": anova, "tukey_sig_pairs": n_sig, "tukey_pairs": len(tk_df),
    "kruskal": {"H": float(H), "p": float(pk), "epsilon_squared": float(eps2)},
    "levene": {"W": float(lev_W), "p": float(lev_p)},
    "chi2_mode_genre": {"chi2": float(chi2_g), "p": float(p_g), "dof": int(dof_g), "cramers_v": v_g,
                        "major_pct": (ct_g["major"] / ct_g.sum(1) * 100).to_dict()},
    "chi2_mode_decade": {"chi2": float(chi2_d), "p": float(p_d), "dof": int(dof_d), "cramers_v": v_d,
                         "major_pct": {int(k): float(v) for k, v in (ct_d["major"] / ct_d.sum(1) * 100).items()}},
    "feature_ranks": ranks.to_dict(), "rank_spearman": rank_corr.to_dict(),
    "top3_anova": fs.sort_values("anova_F", ascending=False).index[:3].tolist(),
    "top3_rfe": fs.sort_values("rfe_rank").index[:3].tolist(),
    "top3_mi": fs.sort_values("mutual_info", ascending=False).index[:3].tolist(),
})

C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\2990580197.py:6: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  "major_pct": (ct_g["major"] / ct_g.sum(1) * 100).to_dict()},
C:\Users\nithi\AppData\Local\Temp\ipykernel_27832\2990580197.py:8: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  "major_pct": {int(k): float(v) for k, v in (ct_d["major"] / ct_d.sum(1) * 100).items()}},
